# Voix Darija pour Shadowing Studio — entraînement Piper

Réentraîne la voix arabe **Kareem** (Piper, qualité *medium*) sur **3 h 20 de darija** (voix M1, corpus DODa, CC BY 4.0).

**Réglages Kaggle (panneau de droite) :** Accelerator = **GPU** (T4 ou P100) · Internet = **On**.
Puis **Save Version → Save & Run All (Commit)** : tout tourne seul (~11 h), même ordinateur éteint.
À la fin, onglet **Output** : télécharger `ary_MA-darija_m1-medium.onnx` et `ary_MA-darija_m1-medium.onnx.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Outil d'entraînement Piper (version testée) + dépendances
!git clone -q https://github.com/OHF-voice/piper1-gpl.git /tmp/piper1-gpl
%cd /tmp/piper1-gpl
!git checkout -q 619f513580c04dff5ef9a680f6f3f6979ed94d2e
!pip install -q "lightning>=2,<3" "jsonargparse[signatures]>=4.27.7" "pysilero-vad>=2.1,<3" "cython>=3,<4" "librosa<1" "tensorboardX>=2,<3" onnx onnxscript onnxruntime pathvalidate
!cd src/piper/train/vits/monotonic_align && mkdir -p monotonic_align && cythonize -q -i core.pyx && mv core*.so monotonic_align/
# Export ONNX : ancien exporteur (le nouveau échoue sur ce modèle)
!sed -i 's/        verbose=False,$/        verbose=False,\n        dynamo=False,/' src/piper/train/export_onnx.py

In [ ]:
# 2. Données darija (préparées pour l'app) + voix de départ Kareem
!mkdir -p /tmp/data && wget -q -O /tmp/data.tar https://huggingface.co/datasets/maromoya/darija-m1-piper/resolve/main/darija_m1_dataset.tar && tar -xf /tmp/data.tar -C /tmp/data && rm /tmp/data.tar
!wget -q -O /tmp/kareem.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ar/ar_JO/kareem/medium/epoch=5079-step=1682020.ckpt"
!wc -l /tmp/data/metadata.csv && ls /tmp/data/wav | wc -l && ls -la /tmp/kareem.ckpt

In [ ]:
# 3. Entraînement (~10 h 15 au maximum, puis arrêt propre)
!cd /tmp/piper1-gpl && PYTHONPATH=src python -m piper.train fit \
  --data.voice_name ary_MA-darija_m1-medium \
  --data.csv_path /tmp/data/metadata.csv --data.audio_dir /tmp/data/wav \
  --data.dataset_type phoneme_ids --data.phoneme_type text --data.espeak_voice ar \
  --model.sample_rate 22050 \
  --data.cache_dir /tmp/cache --data.config_path /kaggle/working/ary_MA-darija_m1-medium.onnx.json \
  --data.batch_size 24 --data.validation_split 0.02 --data.num_test_examples 0 --data.num_workers 4 \
  --model.warmstart_ckpt /tmp/kareem.ckpt \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs 100000 --trainer.max_time 00:10:15:00 \
  --trainer.default_root_dir /tmp/train 2>&1 | grep -v -i warning | tail -n 40

In [ ]:
# 4. Export du fichier de voix pour l'app
import glob, json, os, shutil, subprocess
cks = sorted(glob.glob('/tmp/train/**/last.ckpt', recursive=True), key=os.path.getmtime)
print(cks)
ck = cks[-1]
subprocess.run(['python', '-m', 'piper.train.export_onnx', '--checkpoint', ck,
                '--output-file', '/kaggle/working/ary_MA-darija_m1-medium.onnx'],
               cwd='/tmp/piper1-gpl', env={**os.environ, 'PYTHONPATH': 'src'}, check=True)
cfg_path = '/kaggle/working/ary_MA-darija_m1-medium.onnx.json'
cfg = json.load(open(cfg_path))
cfg.update({'dataset': 'darija_m1', 'phoneme_type': 'espeak', 'espeak': {'voice': 'ar'},
            'language': {'code': 'ary_MA', 'family': 'ar', 'region': 'MA', 'name_english': 'Moroccan Arabic (Darija)'}})
cfg['audio'] = {'sample_rate': 22050, 'quality': 'medium'}
json.dump(cfg, open(cfg_path, 'w'), ensure_ascii=False, indent=2)
shutil.copy(ck, '/kaggle/working/last.ckpt')  # pour reprendre l'entraînement plus tard si besoin
!ls -la /kaggle/working

In [ ]:
# 5. Trois phrases d'écoute (sons tirés des données)
import numpy as np, onnxruntime as ort, soundfile as sf
sess = ort.InferenceSession('/kaggle/working/ary_MA-darija_m1-medium.onnx')
rows = open('/tmp/data/metadata.csv', encoding='utf-8').read().splitlines()[-3:]
for i, row in enumerate(rows, 1):
    ids = np.array([[int(x) for x in row.split('|')[-1].split()]], dtype=np.int64)
    out = sess.run(None, {'input': ids, 'input_lengths': np.array([ids.shape[1]], dtype=np.int64),
                          'scales': np.array([0.667, 1.0, 0.8], dtype=np.float32)})[0]
    sf.write(f'/kaggle/working/ecoute_{i}.wav', out.squeeze(), 22050)
    print(i, row.split('|')[1])